# MiniFASNetV2 Vanilla vs frozen CSMR-v1 — seed 100

Paired backbone-transfer benchmark on frozen Stage-0 5K/2K seed42 manifest and common LCC external-development set. Both branches start from an identical PAD-pretrained MiniFASNetV2 state except a freshly initialized task classifier. The only training-method difference is frozen CSMR-v1. Select checkpoint and threshold on CelebA micro-Val only; LCC is evaluated after both branches freeze. Recover the already-opened Test-2000 identities with the original seed and verify their recorded ordered fingerprint; rebuild the Test-only E1 SCRFD bbox cache after persisting Val/LCC decisions. Test remains a fixed descriptive benchmark.

Input contract: Stage-0 SCRFD face **coordinates**; MiniFASNet official 2.7 crop geometry; OpenCV **BGR** order; direct 80×80 resize; float `[0,1]`; no MobileNet/CelebA mean/std. The source and checkpoint are the local `antispoof/models` files, or exact mounted copies of those files. Upstream crop reference: `minivision-ai/Silent-Face-Anti-Spoofing/src/generate_patches.py`; upstream inference reference: `src/anti_spoof_predict.py`. The notebook uses no upstream downloads or code at runtime.

Test recovery prerequisites on Kaggle: the SCRFD and MiniFASNet datasets are referenced by explicit paths in the config cell; install `insightface==2.0` and a compatible `onnxruntime` before running the notebook. The original completed `csmr-seed100.ipynb` recorded freeze PASS and ordered Test-2000 fingerprint `df3416273c3e7d8bf0caa3b010454ee73236211e034ec12bfb481f056fe237b8`. Reconstruction stops if this fingerprint differs. A rebuilt cache may have a different file SHA from the historical cache; both current branches share the newly saved cache. No prior Kaggle output directory is required.


In [6]:
# KAGGLE DATA ROOTS / FROZEN PAIRED CONFIG
STAGE0_RESOURCE_DIR=("/kaggle/input/datasets/maithanhphuong/" "face-auth-stage0-micro-v1")
CELEBA_ROOT=("/kaggle/input/datasets/attentionlayer241/" "celeba-spoof-for-face-antispoofing/" "CelebA_Spoof_/CelebA_Spoof")
LCC_FASD_ROOT=("/kaggle/input/datasets/aleksandrpikul222/" "lcc-fasd/LCC_FASD")
MINIFASNET_SOURCE_PATH="/kaggle/input/datasets/maithanhphuong/minifasnet/MiniFASNet.py"
MINIFASNET_CHECKPOINT_PATH="/kaggle/input/datasets/maithanhphuong/minifasnet/2.7_80x80_MiniFASNetV2.pth"
SCRFD_MODEL_PATH="/kaggle/input/datasets/maithanhphuong/scrfd-model/det_500m.onnx"
# InsightFace runtime location; SCRFD_MODEL_PATH is copied here after SHA verification.
SCRFD_MODEL_ROOT="/root/.insightface"
OUTPUT_ROOT="/kaggle/working/minifasnetv2_csmr_seed100"
TRAIN_SEED=100
BATCH_SIZE=128  # Only 64 if OOM; rerun both branches from a clean session.
NUM_WORKERS=4
AMP=True
MAX_EPOCHS=12
EARLY_STOP_PATIENCE=3
EARLIEST_EARLY_STOP_EPOCH=6
BACKBONE_LR=1e-4
CLASSIFIER_LR=1e-3
WEIGHT_DECAY=1e-4
LABEL_SMOOTHING=0.1
GRAD_CLIP=5.0
CROP_SCALE=2.7
CSMR_WARMUP_EPOCHS=1
CSMR_LAMBDA=.20
CSMR_BAND_CENTERS=[.15,.45,.75]
CSMR_BAND_SIGMA=.10
CSMR_GAIN_RANGE=[.65,.90]
CSMR_PRESERVE_DC=True
SELECTION_CHUNK_SIZE=32


In [7]:
import csv, hashlib, importlib.util, json, math, os, random, shutil, time, zipfile, copy
from collections import Counter, OrderedDict
from pathlib import Path
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

INPUT_SIZE=80
CELEBA_BBOX_STATUS_FACTORS={"VALID":1.55,"FALLBACK_DET_FAIL":1.50,"FALLBACK_SUSPICIOUS":1.50}

def find_stage0_dir():
    root=Path(STAGE0_RESOURCE_DIR).expanduser()
    if (root/'stage0_summary.json').is_file(): return root.resolve()
    if (root/'stage0'/'stage0_summary.json').is_file(): return (root/'stage0').resolve()
    raise FileNotFoundError(f'Stage 0 bundle missing under {root}')

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_text(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def validate_bbox_record(record, domain, key):
    if not isinstance(record, dict):
        raise TypeError("BBox record must be a mapping for {}".format(key))
    status = str(record.get("status", ""))
    if domain == "celeba":
        if status == "INVALID":
            raise RuntimeError("Selected micro key has INVALID E1 bbox record: {}".format(key))
        if status not in CELEBA_BBOX_STATUS_FACTORS:
            raise RuntimeError("Unknown/unusable E1 bbox status {} for {}".format(status, key))
        expected_factor = CELEBA_BBOX_STATUS_FACTORS[status]
    elif domain == "lcc":
        if status != "VALID":
            raise RuntimeError("Only Stage 0 LCC VALID records may be scored: {} ({})".format(key, status))
        expected_factor = 1.55
    else:
        raise ValueError("Unknown PAD crop domain: {}".format(domain))
    if "crop_factor" not in record:
        raise KeyError("BBox record is missing crop_factor for {}".format(key))
    try:
        crop_factor = float(record["crop_factor"])
    except (TypeError, ValueError) as exc:
        raise ValueError("Invalid crop_factor for {}: {!r}".format(key, record.get("crop_factor"))) from exc
    if not np.isfinite(crop_factor) or not np.isclose(crop_factor, expected_factor, rtol=0.0, atol=1e-6):
        raise ValueError(
            "BBox status/crop-factor mismatch for {}: status={} factor={} expected={}".format(
                key, status, crop_factor, expected_factor
            )
        )
    bbox = np.asarray(record.get("bbox_xyxy", []), dtype=np.float64)
    if bbox.shape != (4,) or not np.isfinite(bbox).all() or bbox[2] <= bbox[0] or bbox[3] <= bbox[1]:
        raise ValueError("Invalid bbox_xyxy for {}: {!r}".format(key, record.get("bbox_xyxy")))
    return crop_factor

def ordered_path_fingerprint(paths):
    return sha256_text("\n".join(map(str, paths)))

def assert_ordered_keys(actual, expected, name):
    actual = list(map(str, actual))
    expected = list(map(str, expected))
    if actual != expected:
        raise RuntimeError(
            "{} sample keys/order differ from the frozen manifest: actual_n={}, expected_n={}".format(
                name, len(actual), len(expected)
            )
        )

def labels_from_attack_codes(codes):
    codes = np.asarray(codes, dtype=np.int64)
    labels = np.full(codes.shape, 2, dtype=np.int64)
    labels[codes == 0] = 0
    labels[np.isin(codes, np.arange(1, 8))] = 1
    return labels

def load_stage0_resources():
    stage0 = find_stage0_dir()
    manifest_path = stage0 / "micro_manifest_v1_seed42.npz"
    bbox_path = stage0 / "celeba_micro_bbox_cache.json"
    lcc_manifest_path = stage0 / "lcc_external_dev_manifest.csv"
    lcc_cache_path = stage0 / "lcc_scrfd_cache_micro_v1.json"
    fingerprints_path = stage0 / "micro_resource_fingerprints.json"
    summary_path = stage0 / "stage0_summary.json"
    for path in [manifest_path, bbox_path, lcc_manifest_path, lcc_cache_path, fingerprints_path, summary_path]:
        if not path.is_file():
            raise FileNotFoundError("Stage 0 resource missing: {}".format(path))
    fingerprints = json.loads(fingerprints_path.read_text())
    stage0_summary = json.loads(summary_path.read_text())
    if stage0_summary.get("schema") != "pad_micro_stage0_summary_v1":
        raise RuntimeError("Unsupported Stage 0 summary schema")
    for path, key in [
        (manifest_path, "micro_manifest_sha256"),
        (lcc_manifest_path, "lcc_manifest_sha256"),
        (lcc_cache_path, "lcc_cache_sha256"),
        (bbox_path, "celeba_bbox_cache_sha256"),
    ]:
        if sha256_file(path) != fingerprints.get(key):
            raise RuntimeError("Stage 0 SHA256 mismatch for {}".format(key))
    with np.load(manifest_path, allow_pickle=False) as pack:
        train_keys = pack["micro_train_keys"].astype(str)
        val_keys = pack["micro_val_keys"].astype(str)
        train_labels = pack["micro_train_labels"].astype(np.int64)
        val_labels = pack["micro_val_labels"].astype(np.int64)
        train_codes = pack["micro_train_attack_codes"].astype(np.int64)
        val_codes = pack["micro_val_attack_codes"].astype(np.int64)
    if len(train_keys) != 5000 or len(val_keys) != 2000:
        raise RuntimeError("Stage 0 must freeze exactly 5,000 Train and 2,000 Val keys")
    if len(set(train_keys) & set(val_keys)):
        raise RuntimeError("Stage 0 Train/Val keys overlap")
    for name, labels, codes in [
        ("Train", train_labels, train_codes),
        ("Val", val_labels, val_codes),
    ]:
        if labels.shape != codes.shape or set(map(int, labels)) != {0, 1, 2}:
            raise RuntimeError("Stage 0 {} label/code arrays are malformed or miss a class".format(name))
        if not np.array_equal(labels, labels_from_attack_codes(codes)):
            raise RuntimeError("Stage 0 {} labels do not match E1 v5.3 attack-code mapping".format(name))
    bbox_payload = json.loads(bbox_path.read_text())
    bbox_cache = bbox_payload.get("records", {})
    selected_keys = set(map(str, train_keys)) | set(map(str, val_keys))
    if bbox_payload.get("schema") != "celeba_micro_bbox_cache_v1" or set(bbox_cache) != selected_keys:
        raise RuntimeError("Stage 0 CelebA bbox cache keys/schema differ from the frozen micro manifest")
    for key, record in bbox_cache.items():
        validate_bbox_record(record, "celeba", key)
    lcc_df = pd.read_csv(lcc_manifest_path)
    lcc_payload = json.loads(lcc_cache_path.read_text())
    lcc_records = lcc_payload.get("records", {})
    if lcc_payload.get("schema") != "pad_micro_lcc_scrfd_cache_v1":
        raise RuntimeError("Unsupported Stage 0 LCC cache schema")
    if lcc_df["path"].astype(str).duplicated().any():
        raise RuntimeError("Stage 0 LCC manifest contains duplicate paths")
    if not set(lcc_df["label"].astype(int)).issubset({0, 1}):
        raise RuntimeError("Stage 0 LCC labels must be exactly 0=Real or 1=Attack")
    record_by_path = {}
    for record in lcc_records.values():
        path = str(record.get("path", ""))
        record_by_path.setdefault(path, []).append(record)
    for row in lcc_df.itertuples(index=False):
        matches = record_by_path.get(str(row.path), [])
        if len(matches) != 1:
            raise RuntimeError("Stage 0 LCC cache join for {} has {} rows".format(row.path, len(matches)))
        record = matches[0]
        if str(record.get("status", "")) != str(row.status):
            raise RuntimeError("Stage 0 LCC manifest/cache status mismatch for {}".format(row.path))
        if int(record.get("label", -1)) != int(row.label):
            raise RuntimeError("Stage 0 LCC manifest/cache label mismatch for {}".format(row.path))
        if str(record.get("frame_id", "")) != str(row.record_id):
            raise RuntimeError("Stage 0 LCC manifest record_id mismatch for {}".format(row.path))
        if str(row.status) == "VALID":
            validate_bbox_record(record, "lcc", row.path)
    valid_paths = lcc_df.loc[lcc_df["status"].astype(str) == "VALID", "path"].astype(str).tolist()
    valid_labels = lcc_df.loc[lcc_df["status"].astype(str) == "VALID", "label"].astype(int).tolist()
    if not valid_paths or set(valid_labels) != {0, 1}:
        raise RuntimeError("Common LCC VALID set must be non-empty and contain Real plus Attack")
    if len(lcc_df) != int(fingerprints.get("lcc_candidate_n", -1)):
        raise RuntimeError("Stage 0 LCC candidate count mismatch")
    if len(valid_paths) != int(fingerprints.get("lcc_valid_face_n", -1)):
        raise RuntimeError("Stage 0 LCC valid-face count mismatch")
    valid_path_sha = ordered_path_fingerprint(valid_paths)
    if valid_path_sha != fingerprints.get("lcc_valid_path_fingerprint_sha256"):
        raise RuntimeError("Stage 0 ordered LCC VALID path fingerprint mismatch")
    if stage0_summary.get("lcc_valid_path_fingerprint_sha256") != valid_path_sha:
        raise RuntimeError("Stage 0 summary ordered LCC VALID path fingerprint mismatch")
    return {
        "stage0_dir": stage0,
        "train_keys": train_keys,
        "val_keys": val_keys,
        "train_labels": train_labels,
        "val_labels": val_labels,
        "train_codes": train_codes,
        "val_codes": val_codes,
        "bbox_cache": bbox_cache,
        "lcc_df": lcc_df,
        "lcc_cache": lcc_records,
        "lcc_valid_paths": valid_paths,
        "fingerprints": fingerprints,
    }

def assert_raw_image_paths(resources, celeba_root, lcc_root):
    celeba_keys = list(map(str, resources["train_keys"])) + list(map(str, resources["val_keys"]))
    lcc_paths = resources["lcc_df"]["path"].astype(str).tolist()
    missing_celeba = [str(Path(celeba_root) / key) for key in celeba_keys if not (Path(celeba_root) / key).is_file()]
    missing_lcc = [str(Path(lcc_root) / path) for path in lcc_paths if not (Path(lcc_root) / path).is_file()]
    if missing_celeba or missing_lcc:
        raise FileNotFoundError(
            "Raw PAD image preflight failed before training: CelebA missing={}, LCC missing={}".format(
                missing_celeba[:5], missing_lcc[:5]
            )
        )
    print("Verified raw image paths before training: CelebA selected={}, LCC candidates={}".format(
        len(celeba_keys), len(lcc_paths)
    ))

def bbox_jitter(record):
    x1, y1, x2, y2 = map(float, record["bbox_xyxy"])
    w, h = x2 - x1, y2 - y1
    if random.random() < 0.20:
        base = max(w, h)
        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        scale = random.uniform(0.95, 1.05)
        cx += random.uniform(-0.05, 0.05) * base
        cy += random.uniform(-0.05, 0.05) * base
        w, h = w * scale, h * scale
        return [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]
    return [x1, y1, x2, y2]

def apply_e1_mild_augmentation(rgb):
    if random.random() < 0.5:
        rgb = cv2.flip(rgb, 1)
    if random.random() < 0.30:
        alpha = random.uniform(0.90, 1.10)
        beta = random.uniform(-0.10, 0.10) * 255.0
        rgb = np.clip(rgb.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    return rgb

def binary_score(logits):
    logits = torch.as_tensor(logits)
    if logits.ndim != 2 or logits.shape[1] != 3:
        raise ValueError("PAD logits must have shape [N, 3] for Real/Physical/Digital")
    if not torch.isfinite(logits).all():
        raise ValueError("PAD logits contain NaN or Infinity")
    return logits[:, 0] - torch.logsumexp(logits[:, 1:], dim=1)

def safe_auc(y_real, scores):
    from sklearn.metrics import roc_auc_score
    y_real = np.asarray(y_real, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y_real.ndim != 1 or scores.ndim != 1 or len(y_real) != len(scores) or len(y_real) == 0:
        raise ValueError("AUC needs non-empty one-dimensional labels and scores of equal length")
    if not np.isfinite(scores).all():
        raise ValueError("AUC scores contain NaN or Infinity")
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("AUC requires both binary classes (positive class is Real)")
    return float(roc_auc_score(y_real, scores))

def threshold_from_val(y3, scores):
    y3 = np.asarray(y3, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y3.ndim != 1 or scores.ndim != 1 or len(y3) != len(scores) or len(y3) == 0:
        raise ValueError("Threshold calibration needs non-empty 1-D labels/scores with equal lengths")
    if not set(map(int, np.unique(y3))).issubset({0, 1, 2}):
        raise ValueError("CelebA labels must be 0=Real, 1=Physical, 2=Digital")
    y_real = (y3 == 0).astype(np.int64)
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("Threshold calibration requires at least one Real and one Attack sample")
    if not np.isfinite(scores).all():
        raise ValueError("Threshold calibration scores contain NaN or Infinity")
    values = np.unique(scores)
    lower = np.nextafter(values[0], -np.inf)
    upper = np.nextafter(values[-1], np.inf)
    if not np.isfinite(lower):
        lower = values[0]
    if not np.isfinite(upper):
        upper = values[-1]
    midpoints = values[:-1] / 2.0 + values[1:] / 2.0
    candidates = np.concatenate(([lower], midpoints, [upper]))
    best = None
    for threshold in candidates:
        pred_real = scores >= threshold
        real = y_real == 1
        attack = y_real == 0
        bpcer = float(np.mean(~pred_real[real]))
        apcer = float(np.mean(pred_real[attack]))
        acer = 0.5 * (apcer + bpcer)
        key = (acer, abs(float(threshold)), float(threshold))
        if best is None or key < best[0]:
            best = (key, float(threshold))
    if best is None or not np.isfinite(best[1]):
        raise RuntimeError("Threshold search did not produce a finite threshold")
    return best[1]

def metrics_binary(y3, scores, threshold, coverage=None, candidate_n=None):
    y3 = np.asarray(y3, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y3.ndim != 1 or scores.ndim != 1 or len(y3) != len(scores) or len(y3) == 0:
        raise ValueError("PAD metrics need non-empty 1-D labels/scores with equal lengths")
    if not set(map(int, np.unique(y3))).issubset({0, 1, 2}):
        raise ValueError("PAD labels must be from the frozen class mapping 0/1/2")
    if not np.isfinite(scores).all() or not np.isfinite(float(threshold)):
        raise ValueError("PAD scores and threshold must be finite")
    y_real = (y3 == 0).astype(np.int64)
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("PAD metrics require at least one Real and one Attack sample")
    if coverage is not None and (not np.isfinite(float(coverage)) or not 0.0 <= float(coverage) <= 1.0):
        raise ValueError("Detector coverage must be finite and within [0, 1]")
    scored_n = len(y3)
    candidate_n = int(candidate_n if candidate_n is not None else scored_n)
    if candidate_n < scored_n:
        raise ValueError("candidate_n cannot be less than n_scored")
    pred_real = scores >= float(threshold)
    real = y_real == 1
    attack = y_real == 0
    bpcer = float(np.mean(~pred_real[real]))
    apcer = float(np.mean(pred_real[attack]))
    acer = 0.5 * (apcer + bpcer)
    return {
        "n_scored": int(scored_n),
        "candidate_n": candidate_n,
        "detector_coverage": float(coverage if coverage is not None else 1.0),
        "apcer": apcer,
        "bpcer": bpcer,
        "acer": acer,
        "hter": acer,
        "auc": safe_auc(y_real, scores),
        "accuracy": float(np.mean(pred_real == y_real)),
        "locked_logit_threshold": float(threshold),
    }

def metrics_celeba(y3, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    result = metrics_binary(y3, scores, threshold, coverage=1.0, candidate_n=len(y3))
    result["accuracy_3class"] = float(np.mean(np.argmax(logits, axis=1) == np.asarray(y3)))
    return result, scores

def make_prediction_frame(keys, y3, codes, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    pred_real = scores >= threshold
    result = pd.DataFrame({
        "key": list(map(str, keys)),
        "true_class": np.asarray(y3, dtype=np.int64),
        "attack_code": np.asarray(codes, dtype=np.int64),
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
        "pred_real": pred_real,
        "correct_binary": pred_real == (np.asarray(y3) == 0),
        "pred_class_3": np.argmax(logits, axis=1),
        "correct_3class": np.argmax(logits, axis=1) == np.asarray(y3),
    })
    return result

def eval_spatial(model, loader, device, amp):
    model.eval()
    logits_all, y_all, keys_all, codes_all = [], [], [], []
    with torch.no_grad():
        for x, y, keys, codes in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
            codes_all.extend(np.asarray(codes, dtype=np.int64).tolist())
    if not logits_all:
        raise RuntimeError("Spatial evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    labels = np.asarray(y_all, dtype=np.int64)
    codes = np.asarray(codes_all, dtype=np.int64)
    if len(logits) != len(labels) or len(keys_all) != len(labels) or len(codes) != len(labels):
        raise RuntimeError("Spatial evaluation outputs have inconsistent sample counts")
    return logits, labels, list(map(str, keys_all)), codes

def eval_lcc_spatial(model, loader, device, amp, lcc_frame, threshold):
    model.eval()
    logits_all, y_all, keys_all = [], [], []
    with torch.no_grad():
        for x, y, keys, _status in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
    if not logits_all:
        raise RuntimeError("Common LCC VALID evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    y = np.asarray(y_all, dtype=np.int64)
    keys = list(map(str, keys_all))
    expected_paths = lcc_frame.loc[lcc_frame["status"].astype(str) == "VALID", "path"].astype(str).tolist()
    assert_ordered_keys(keys, expected_paths, "Stage 1 LCC")
    if len(logits) != len(y) or len(keys) != len(y):
        raise RuntimeError("Stage 1 LCC logits/labels/keys have inconsistent counts")
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    scored = pd.DataFrame({
        "path": keys, "label": y,
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
    })
    full = lcc_frame[["path", "label", "status"]].merge(
        scored, on=["path", "label"], how="left", validate="one_to_one"
    )
    full["dataset_role"] = "LCC-FASD external-dev / cross-domain stress set"
    valid = full["status"].astype(str).eq("VALID")
    if full.loc[valid, "pad_logit_score"].isna().any() or int(valid.sum()) != len(keys):
        raise RuntimeError("Stage 1 inference failed to score the exact common LCC VALID set")
    full["pred_real"] = np.nan
    full.loc[valid, "pred_real"] = full.loc[valid, "pad_logit_score"] >= float(threshold)
    full["pred_real"] = full["pred_real"].where(valid, np.nan)
    return logits, y, keys, full

def write_zip(folder, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(Path(folder).rglob("*")):
            if path.is_file() and path.resolve() != Path(zip_path).resolve():
                archive.write(path, arcname=Path(Path(folder).name) / path.relative_to(folder))


In [8]:
from datetime import datetime, timezone
RUN_STARTED=time.monotonic()
def log(section,message):
    print(f'[{datetime.now(timezone.utc).isoformat(timespec="seconds")} +{time.monotonic()-RUN_STARTED:.1f}s {section}] {message}',flush=True)

def resolve_project_file(explicit,relative):
    candidates=[Path(explicit).expanduser()] if explicit else [Path.cwd()/relative,Path('/kaggle/working')/relative]
    found=[p.resolve() for p in candidates if p.is_file()]
    if len(found)!=1:
        raise FileNotFoundError(f'Expected one {relative}; set its exact mounted path. Checked {candidates}')
    return found[0]

def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Official CropImage._get_new_box geometry, adapted from xyxy face coordinates.
    src_h,src_w=image.shape[:2]
    x1,y1,x2,y2=map(float,bbox_xyxy)
    box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:
        raise ValueError('Invalid MiniFASNet image/bbox geometry')
    used_scale=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale))
    new_w,new_h=box_w*used_scale,box_h*used_scale
    center_x,center_y=box_w/2+x1,box_h/2+y1
    left,top=center_x-new_w/2,center_y-new_h/2
    right,bottom=center_x+new_w/2,center_y+new_h/2
    if left<0: right-=left;left=0
    if top<0: bottom-=top;top=0
    if right>src_w-1: left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1: top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:
        raise RuntimeError(f'MiniFASNet crop escaped image: {(lx,ty,rx,by)}')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty MiniFASNet crop')
    return cv2.resize(patch,(size,size)),(lx,ty,rx,by),used_scale

def mini_tensor(patch):
    if patch.shape!=(INPUT_SIZE,INPUT_SIZE,3) or patch.dtype!=np.uint8:
        raise ValueError('Expected 80x80 uint8 BGR patch')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)

class MiniCelebADataset(Dataset):
    def __init__(self,keys,labels,codes,root,records,train=False):
        self.keys=list(map(str,keys));self.labels=np.asarray(labels,dtype=np.int64)
        self.codes=np.asarray(codes,dtype=np.int64);self.root=Path(root);self.records=records;self.train=train
        if len(self.keys)!=len(self.labels) or len(self.keys)!=len(self.codes):raise ValueError('Key/label/code lengths differ')
        missing=[k for k in self.keys if k not in records]
        if missing:raise RuntimeError(f'{len(missing)} CelebA keys lack frozen face bbox')
    def __len__(self):return len(self.keys)
    def __getitem__(self,i):
        key=self.keys[i];rec=self.records[key]
        validate_bbox_record(rec,'celeba',key)  # validate E1 provenance/factor, then do NOT use E1 crop factor
        image=cv2.imread(str(self.root/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(self.root/key)
        bbox=bbox_jitter(rec) if self.train else rec['bbox_xyxy']
        patch,_,_=mini_crop_bgr(image,bbox)
        if self.train:patch=apply_e1_mild_augmentation(patch)
        return mini_tensor(patch),int(self.labels[i]),key,int(self.codes[i])

class MiniLCCDataset(Dataset):
    def __init__(self,frame,root,records):
        self.root=Path(root);self.rows=[]
        by_path={}
        for rec in records.values():by_path.setdefault(str(rec.get('path','')),[]).append(rec)
        for row in frame.itertuples(index=False):
            if str(row.status)!='VALID':continue
            candidates=by_path.get(str(row.path),[])
            if len(candidates)!=1:raise RuntimeError(f'LCC bbox join differs for {row.path}')
            validate_bbox_record(candidates[0],'lcc',row.path)
            self.rows.append((str(row.path),int(row.label),candidates[0]))
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        key,label,rec=self.rows[i]
        image=cv2.imread(str(self.root/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(self.root/key)
        patch,_,_=mini_crop_bgr(image,rec['bbox_xyxy'])
        return mini_tensor(patch),label,key,'VALID'

def source_module(path):
    spec=importlib.util.spec_from_file_location('local_minifasnet_v2',path)
    if spec is None or spec.loader is None:raise RuntimeError('Cannot import local MiniFASNet.py')
    module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
    return module

def new_model(module):
    return module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),
        drop_p=.2,num_classes=3,img_channel=3)

def state_fingerprint(state):
    h=hashlib.sha256()
    for key in sorted(state):
        value=state[key].detach().cpu().contiguous()
        h.update(key.encode());h.update(str(value.dtype).encode());h.update(str(tuple(value.shape)).encode())
        h.update(value.numpy().tobytes())
    return h.hexdigest()

def load_pretrained_canonical(module,checkpoint_path):
    raw=torch.load(checkpoint_path,map_location='cpu',weights_only=True)
    if not isinstance(raw,OrderedDict) or not raw or not all(k.startswith('module.') for k in raw):
        raise RuntimeError('Expected official OrderedDict with module. prefix')
    stripped=OrderedDict((k[7:],v) for k,v in raw.items())
    if list(stripped['prob.weight'].shape)!=[3,128]:raise RuntimeError('Official prob.weight shape differs from [3,128]')
    if tuple(stripped['conv_6_dw.conv.weight'].shape[-2:])!=(5,5):raise RuntimeError('Official conv6 kernel is not 5x5')
    seed_everything(TRAIN_SEED)
    model=new_model(module)
    model_keys=model.state_dict()
    skipped=['prob.weight']
    shape_mismatch=[k for k,v in stripped.items() if k in model_keys and tuple(v.shape)!=tuple(model_keys[k].shape) and k not in skipped]
    unexpected=[k for k in stripped if k not in model_keys]
    missing_pretrained=[k for k in model_keys if k not in stripped and k not in skipped]
    if shape_mismatch or unexpected or missing_pretrained:raise RuntimeError(f'Unrelated checkpoint mismatch: shape={shape_mismatch[:5]} unexpected={unexpected[:5]} missing={missing_pretrained[:5]}')
    compatible=OrderedDict((k,v) for k,v in stripped.items() if k not in skipped)
    missing,unexpected_load=model.load_state_dict(compatible,strict=False)
    if set(missing)!={'prob.weight'} or unexpected_load:raise RuntimeError(f'Loading audit failed: missing={missing}, unexpected={unexpected_load}')
    model.prob=nn.Linear(128,3,bias=False)  # new task semantics, deterministic seed 100
    if not model.prob.weight.requires_grad or not model.conv1.conv.weight.requires_grad:
        raise RuntimeError('Classifier/backbone must remain trainable')
    audit={'schema':'mini_fasnet_pretrained_loading_audit_v1','checkpoint_file':checkpoint_path.name,
        'checkpoint_sha256':sha256_file(checkpoint_path),'checkpoint_bytes':checkpoint_path.stat().st_size,
        'architecture_source_sha256':sha256_file(MINIFASNET_SOURCE),
        'checkpoint_keys':len(raw),'loaded_pretrained_key_count':len(compatible),
        'missing_keys':list(missing),'unexpected_keys':list(unexpected_load),
        'intentionally_skipped_keys':skipped,'shape_mismatch_keys':shape_mismatch,
        'input_size':[80,80],'crop_scale':CROP_SCALE,'channel_order':'BGR',
        'conv6_kernel':[5,5],'embedding_size':128,'classifier_shape':[3,128],
        'total_params':sum(p.numel() for p in model.parameters()),
        'trainable_params':sum(p.numel() for p in model.parameters() if p.requires_grad)}
    return model,audit

def save_preprocess_gallery(resources,root,out_path):
    rng=np.random.default_rng(100)
    fig,axes=plt.subplots(3,3,figsize=(11,11))
    for cls in range(3):
        pool=np.where(resources['train_labels']==cls)[0]
        idx=int(rng.choice(pool));key=str(resources['train_keys'][idx]);rec=resources['bbox_cache'][key]
        image=cv2.imread(str(Path(root)/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(Path(root)/key)
        patch,box,used_scale=mini_crop_bgr(image,rec['bbox_xyxy'])
        raw=cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
        p1=tuple(map(int,rec['bbox_xyxy'][:2]));p2=tuple(map(int,rec['bbox_xyxy'][2:]))
        cv2.rectangle(raw,p1,p2,(255,0,0),2)
        axes[cls,0].imshow(raw);axes[cls,0].set_title(f'{cls} raw + Stage0 bbox')
        # Show the actual expanded crop before final resize, without changing model input.
        lx,ty,rx,by=box
        axes[cls,1].imshow(cv2.cvtColor(image[ty:by+1,lx:rx+1],cv2.COLOR_BGR2RGB))
        axes[cls,1].set_title(f'2.7 crop (effective {used_scale:.2f})')
        axes[cls,2].imshow(cv2.cvtColor(patch,cv2.COLOR_BGR2RGB))
        axes[cls,2].set_title('80x80 BGR tensor view')
        for ax in axes[cls]:ax.axis('off')
    fig.tight_layout();fig.savefig(out_path,dpi=130);plt.close(fig)


In [9]:
import re
import torch.nn.functional as F
BANDS=('LOW','MID','HIGH')

def normalize_unit(x):
    # MiniFASNet checkpoint expects BGR float [0,1]; no MNV3 mean/std.
    return x

def radial_masks(size=INPUT_SIZE, device='cpu'):
    y = torch.arange(size, device=device, dtype=torch.float32) - size // 2
    yy, xx = torch.meshgrid(y, y, indexing='ij')
    radius = torch.sqrt(xx.square() + yy.square()) / math.sqrt(2 * (size // 2) ** 2)
    centers = torch.as_tensor(CSMR_BAND_CENTERS, dtype=torch.float32, device=device)
    masks = torch.exp(-((radius[None] - centers[:, None, None]) ** 2) /
                      (2 * CSMR_BAND_SIGMA ** 2))
    if masks.shape != (3, size, size) or not torch.isfinite(masks).all() or not ((masks >= 0) & (masks <= 1)).all():
        raise RuntimeError('Invalid CSMR radial masks')
    if len(set(map(float, CSMR_BAND_CENTERS))) != 3:
        raise RuntimeError('CSMR band centers must differ')
    return masks, radius

def spectral_views(x, gains, masks):
    if x.ndim != 4 or x.shape[1:] != (3, INPUT_SIZE, INPUT_SIZE) or not torch.isfinite(x).all():
        raise ValueError('CSMR input must be finite [B,3,224,224]')
    x = x.float()
    gains = gains.to(device=x.device, dtype=torch.float32).reshape(-1)
    if len(gains) != len(x) or not ((gains >= 0) & (gains <= 1)).all():
        raise ValueError('Invalid gains')
    spectrum = torch.fft.fftshift(torch.fft.fft2(x, norm='ortho'), dim=(-2, -1))
    gain_fields = 1 - (1 - gains[:, None, None, None]) * masks[None]
    if CSMR_PRESERVE_DC:
        gain_fields[:, :, INPUT_SIZE // 2, INPUT_SIZE // 2] = 1.0
    perturbed = spectrum[:, None] * gain_fields[:, :, None]
    raw = torch.fft.ifft2(torch.fft.ifftshift(perturbed, dim=(-2, -1)), norm='ortho').real
    clipped = ((raw < 0) | (raw > 1)).float().mean(dim=(2, 3, 4))
    views = raw.clamp(0, 1)
    if not torch.isfinite(views).all():
        raise RuntimeError('Non-finite CSMR intervention')
    return views, gain_fields, clipped

def aligned_margin(logits, labels):
    d = binary_score(logits.float())
    labels = torch.as_tensor(labels, device=d.device)
    if not torch.isin(labels, torch.tensor([0, 1, 2], device=d.device)).all():
        raise ValueError('Invalid PAD label')
    return torch.where(labels == 0, d, -d)

def harmful_margin_loss(clean_margin, worst_margin):
    drop = clean_margin.detach() - worst_margin
    harmful = drop > 0
    per = F.smooth_l1_loss(worst_margin, clean_margin.detach(), reduction='none')
    return (per * harmful.float()).mean(), harmful, drop

def selection_by_margins(margins):
    if margins.ndim != 2 or margins.shape[1] != 3 or not torch.isfinite(margins).all():
        raise ValueError('Expected finite [B,3] margins')
    return margins.argmin(dim=1)

def select_worst_view(model, base, labels, masks, amp, chunk_size):
    if chunk_size < 1:
        raise ValueError('Selection chunk size must be positive')
    was_training = model.training
    model.eval()
    chosen, indexes, all_gains, all_clips = [], [], [], []
    try:
        with torch.no_grad():
            for start in range(0, len(base), chunk_size):
                x = base[start:start+chunk_size]
                y = labels[start:start+chunk_size]
                gains = torch.empty(len(x), device=x.device).uniform_(*CSMR_GAIN_RANGE)
                views, _, clips = spectral_views(x, gains, masks)
                margins = []
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=amp):
                        logits = model(normalize_unit(views[:, band]))
                    margins.append(aligned_margin(logits, y))
                indexes_chunk = selection_by_margins(torch.stack(margins, dim=1))
                ix = torch.arange(len(x), device=x.device)
                chosen.append(views[ix, indexes_chunk].detach())
                indexes.append(indexes_chunk.detach())
                all_gains.append(gains.detach())
                all_clips.append(clips[ix, indexes_chunk].detach())
    finally:
        model.train(was_training)
    return tuple(torch.cat(a) for a in (chosen, indexes, all_gains, all_clips))

def strict_json_load(path):
    def reject(value):
        raise ValueError('Non-finite JSON constant: ' + value)
    return json.loads(Path(path).read_text(), parse_constant=reject)

def strict_json_write(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, allow_nan=False) + '\n')

def intervention_sanity(masks, device):
    gen = torch.Generator(device=device).manual_seed(TRAIN_SEED)
    x = torch.rand((2, 3, INPUT_SIZE, INPUT_SIZE), generator=gen, device=device)
    gain = torch.tensor([0.65, 0.90], device=device)
    views, fields, clips = spectral_views(x, gain, masks)
    assert views.shape == (2, 3, 3, INPUT_SIZE, INPUT_SIZE)
    assert torch.all(fields[:, :, INPUT_SIZE//2, INPUT_SIZE//2] == 1)
    unity, _, _ = spectral_views(x, torch.ones(2, device=device), masks)
    assert torch.max(torch.abs(unity[:, 0] - x)).item() < 1e-5
    ft = torch.fft.fftshift(torch.fft.fft2(x, norm='ortho'), dim=(-2,-1))
    modified = ft[:, None] * fields[:, :, None]
    valid = ft.abs()[:, None] > 1e-5
    phase_error = (torch.angle(modified) - torch.angle(ft[:, None])).abs()
    phase_error = torch.minimum(phase_error, 2*math.pi-phase_error)
    assert phase_error.expand_as(valid.expand_as(modified))[valid.expand_as(modified)].max().item() < 1e-4
    report = {}
    for i, band in enumerate(BANDS):
        diff = (views[:, i] - x).abs()
        report[band] = {'mean_absolute_pixel_difference': float(diff.mean()),
                        'max_absolute_pixel_difference': float(diff.max()),
                        'clipped_pixel_fraction': float(clips[:, i].mean())}
        if report[band]['clipped_pixel_fraction'] > 0.05:
            print('WARNING: clipping exceeds 5% for', band)
    return report


In [10]:
# One fixed backbone-transfer protocol; only paired batch-size 64 OOM rerun is allowed.
if (TRAIN_SEED!=100 or BATCH_SIZE not in (128,64) or MAX_EPOCHS!=12 or
    EARLY_STOP_PATIENCE!=3 or EARLIEST_EARLY_STOP_EPOCH!=6 or
    BACKBONE_LR!=1e-4 or CLASSIFIER_LR!=1e-3 or WEIGHT_DECAY!=1e-4 or
    LABEL_SMOOTHING!=.1 or GRAD_CLIP!=5. or CROP_SCALE!=2.7 or
    CSMR_WARMUP_EPOCHS!=1 or CSMR_LAMBDA!=.20 or
    CSMR_BAND_CENTERS!=[.15,.45,.75] or CSMR_BAND_SIGMA!=.10 or
    CSMR_GAIN_RANGE!=[.65,.90] or CSMR_PRESERVE_DC is not True or
    SELECTION_CHUNK_SIZE<1):
    raise RuntimeError('Frozen MiniFASNet/CSMR paired protocol changed')
if not torch.cuda.is_available():raise RuntimeError('Kaggle GPU required for this paired benchmark')
device=torch.device('cuda');use_amp=bool(AMP)
log('PREFLIGHT',f'GPU {torch.cuda.get_device_name(0)}; AMP={use_amp}')
MINIFASNET_SOURCE=resolve_project_file(MINIFASNET_SOURCE_PATH,'antispoof/models/MiniFASNet.py')
MINIFASNET_CHECKPOINT=resolve_project_file(MINIFASNET_CHECKPOINT_PATH,'antispoof/models/2.7_80x80_MiniFASNetV2.pth')
log('PREFLIGHT',f'Source={MINIFASNET_SOURCE}; checkpoint={MINIFASNET_CHECKPOINT}')
resources=load_stage0_resources()
celeba_root=Path(CELEBA_ROOT)
lcc_root=Path(LCC_FASD_ROOT)
if not (celeba_root/'metas/intra_test/train_label.json').is_file():raise FileNotFoundError(f'Wrong CelebA root: {celeba_root}')
if not lcc_root.is_dir():raise FileNotFoundError(f'Wrong LCC root: {lcc_root}')
assert_raw_image_paths(resources,celeba_root,lcc_root)
log('PREFLIGHT',f'Stage0 manifest seed42 SHA={resources["fingerprints"]["micro_manifest_sha256"]}; Train={len(resources["train_keys"])} Val={len(resources["val_keys"])} LCC valid={len(resources["lcc_valid_paths"])}')
module=source_module(MINIFASNET_SOURCE)
canonical_model,loading_audit=load_pretrained_canonical(module,MINIFASNET_CHECKPOINT)
canonical_state=copy.deepcopy(canonical_model.state_dict())
initial_sha=state_fingerprint(canonical_state)
canonical_model.eval()
with torch.no_grad():
    if canonical_model(torch.rand(2,3,80,80)).shape!=(2,3):raise RuntimeError('MiniFASNetV2 output shape mismatch')
log('PREFLIGHT',f'MiniFASNet checkpoint audit={loading_audit}; initial SHA={initial_sha}')
if abs(loading_audit['total_params']-430000)>100000:log('PREFLIGHT','WARNING: parameter count differs materially from ~0.43M')
masks,_=radial_masks(size=80,device=device)
fft_sanity=intervention_sanity(masks,device)
log('PREFLIGHT',f'CSMR 80x80 FFT sanity={fft_sanity}')
probe_logits=torch.tensor([[4.,0.,0.],[0.,4.,0.],[0.,0.,4.]])
if not torch.all(aligned_margin(probe_logits,torch.tensor([0,1,2]))>0):raise RuntimeError('Aligned margin sign test failed')
zero,_,_=harmful_margin_loss(torch.tensor([1.]),torch.tensor([2.]))
positive,_,_=harmful_margin_loss(torch.tensor([1.]),torch.tensor([0.]))
if float(zero)!=0 or float(positive)<=0:raise RuntimeError('Harmful-only margin loss test failed')
# Check eval/no_grad selection does not touch BN running state and restores train mode.
probe=copy.deepcopy(canonical_model).to(device);probe.train()
before=probe.conv1.bn.running_mean.detach().clone()
with torch.no_grad():
    select_worst_view(probe,torch.rand(2,3,80,80,device=device),
        torch.tensor([0,1],device=device),masks,use_amp,1)
if not probe.training or not torch.equal(before,probe.conv1.bn.running_mean):
    raise RuntimeError('CSMR selection changed MiniFASNet BatchNorm state')
del probe;torch.cuda.empty_cache()
output_root=Path(OUTPUT_ROOT)
if output_root.exists():raise RuntimeError(f'Unsafe existing output directory: {output_root}; start clean')
vanilla_dir=output_root/'vanilla';csmr_dir=output_root/'csmr';comparison_dir=output_root/'comparison'
for folder in (vanilla_dir,csmr_dir,comparison_dir):folder.mkdir(parents=True,exist_ok=False)
strict_json_write(comparison_dir/'model_pretrained_loading_audit.json',loading_audit)
strict_json_write(comparison_dir/'initialization_fingerprint.json',
    {'schema':'mini_fasnet_initialization_v1','training_seed':TRAIN_SEED,
     'canonical_initial_state_sha256':initial_sha,'vanilla_initial_state_sha256':None,
     'csmr_initial_state_sha256':None,'checkpoint_sha256':loading_audit['checkpoint_sha256'],
     'source_sha256':loading_audit['architecture_source_sha256']})
save_preprocess_gallery(resources,celeba_root,comparison_dir/'preprocessing_gallery.png')
fig,axes=plt.subplots(1,3,figsize=(10,3))
for i,b in enumerate(BANDS):
    axes[i].imshow(masks[i].cpu().numpy(),cmap='viridis',vmin=0,vmax=1)
    axes[i].set_title(f'{b}: c={CSMR_BAND_CENTERS[i]}');axes[i].axis('off')
fig.tight_layout();fig.savefig(csmr_dir/'frequency_masks.png',dpi=130);plt.close(fig)
# Diagnostic CSMR gallery, deterministic and independent of training RNG.
rng=np.random.default_rng(100)
train_ds_gallery=MiniCelebADataset(resources['train_keys'],resources['train_labels'],
    resources['train_codes'],celeba_root,resources['bbox_cache'],train=False)
fig,axes=plt.subplots(6,4,figsize=(12,17))
for row,cls in enumerate([0,0,1,1,2,2]):
    pool=np.where(resources['train_labels']==cls)[0]
    idx=int(rng.choice(pool));x,label,key,_=train_ds_gallery[idx]
    gain=float(rng.uniform(*CSMR_GAIN_RANGE))
    views,_,_=spectral_views(x[None].to(device),torch.tensor([gain],device=device),masks)
    for col,item in enumerate([x,*list(views[0].cpu())]):
        axes[row,col].imshow(item.permute(1,2,0).numpy()[...,::-1])
        axes[row,col].set_title(f'{["Clean",*BANDS][col]} class={label} g={gain:.3f}')
        axes[row,col].axis('off')
fig.tight_layout();fig.savefig(csmr_dir/'intervention_gallery.png',dpi=130);plt.close(fig)
log('PREFLIGHT',f'Galleries saved; BGR 80x80, crop 2.7; output={output_root}')


[2026-09-29T13:57:43+00:00 +0.1s PREFLIGHT] GPU Tesla T4; AMP=True
[2026-09-29T13:57:43+00:00 +0.1s PREFLIGHT] Source=/kaggle/input/datasets/maithanhphuong/minifasnet/MiniFASNet.py; checkpoint=/kaggle/input/datasets/maithanhphuong/minifasnet/2.7_80x80_MiniFASNetV2.pth
Verified raw image paths before training: CelebA selected=7000, LCC candidates=3766
[2026-09-29T13:58:32+00:00 +49.5s PREFLIGHT] Stage0 manifest seed42 SHA=7c1eed4abaab7ee193758ea4bf4a0c36b98e43b5fb8d84c5c0e7d424607a0aa3; Train=5000 Val=2000 LCC valid=3762
[2026-09-29T13:58:33+00:00 +50.0s PREFLIGHT] MiniFASNet checkpoint audit={'schema': 'mini_fasnet_pretrained_loading_audit_v1', 'checkpoint_file': '2.7_80x80_MiniFASNetV2.pth', 'checkpoint_sha256': 'a5eb02e1843f19b5386b953cc4c9f011c3f985d0ee2bb9819eea9a142099bec0', 'checkpoint_bytes': 1849453, 'architecture_source_sha256': 'e498c4ec5e1ddfaba62b941a126c19d65aa564999f3309661fe43ee8bf38acd7', 'checkpoint_keys': 334, 'loaded_pretrained_key_count': 333, 'missing_keys': ['prob

/tmp/ipykernel_58/3028455222.py:74: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


[2026-09-29T13:58:41+00:00 +58.1s PREFLIGHT] Galleries saved; BGR 80x80, crop 2.7; output=/kaggle/working/minifasnetv2_csmr_seed100


In [11]:
def run_branch(name,csmr_enabled):
    seed_everything(TRAIN_SEED)
    model=new_model(module)
    model.load_state_dict(canonical_state,strict=True)
    branch_initial_sha=state_fingerprint(model.state_dict())
    if branch_initial_sha!=initial_sha:raise RuntimeError(f'{name} initial model differs from canonical state')
    model=model.to(device)
    log(name,f'Initial SHA {branch_initial_sha}; trainable params={sum(p.numel() for p in model.parameters() if p.requires_grad)}')
    train_ds=MiniCelebADataset(resources['train_keys'],resources['train_labels'],
        resources['train_codes'],celeba_root,resources['bbox_cache'],train=True)
    train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,drop_last=False,
        num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=NUM_WORKERS>0)
    val_ds=MiniCelebADataset(resources['val_keys'],resources['val_labels'],
        resources['val_codes'],celeba_root,resources['bbox_cache'],train=False)
    val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,drop_last=False,
        num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=NUM_WORKERS>0)
    backbone=[p for n,p in model.named_parameters() if not n.startswith('prob.')]
    classifier=list(model.prob.parameters())
    optimizer=torch.optim.AdamW([{'params':backbone,'lr':BACKBONE_LR},
        {'params':classifier,'lr':CLASSIFIER_LR}],weight_decay=WEIGHT_DECAY)
    criterion=nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    history=[];diagnostics=[];class_rows=[]
    best_acer=float('inf');best_epoch=None;best_state=None
    try:
        for epoch in range(1,MAX_EPOCHS+1):
            start=time.monotonic();model.train();losses=[];cls_losses=[];margin_losses=[]
            band_counts=torch.zeros(3,dtype=torch.int64);class_counts=torch.zeros((3,3),dtype=torch.int64)
            class_harmful=torch.zeros(3,dtype=torch.int64);class_drop=torch.zeros(3,dtype=torch.float64)
            class_seen=torch.zeros(3,dtype=torch.int64);harmful_n=0;csmr_n=0
            harmful_drop_sum=clean_margin_sum=worst_margin_sum=gain_sum=clip_sum=0.
            log(name,f'epoch {epoch}/{MAX_EPOCHS} start')
            for batch_index,(x,y,_keys,_codes) in enumerate(train_loader,1):
                x=x.to(device,non_blocking=True);y=y.to(device,non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                if csmr_enabled and epoch>CSMR_WARMUP_EPOCHS:
                    selected,band_idx,gains,clips=select_worst_view(model,x,y,masks,use_amp,SELECTION_CHUNK_SIZE)
                    model.train()
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        logits=model(torch.cat([x,selected],dim=0))
                        clean_logits,worst_logits=logits.chunk(2)
                        cls_loss=.5*criterion(clean_logits,y)+.5*criterion(worst_logits,y)
                        clean_margin=aligned_margin(clean_logits,y)
                        worst_margin=aligned_margin(worst_logits,y)
                        margin_loss,harmful,drop=harmful_margin_loss(clean_margin,worst_margin)
                        loss=cls_loss+CSMR_LAMBDA*margin_loss
                    n=len(y);csmr_n+=n
                    gain_sum+=float(gains.sum().cpu());clip_sum+=float(clips.sum().cpu())
                    harmful_n+=int(harmful.sum().cpu())
                    harmful_drop_sum+=float(drop.clamp_min(0).sum().detach().cpu())
                    clean_margin_sum+=float(clean_margin.sum().detach().cpu())
                    worst_margin_sum+=float(worst_margin.sum().detach().cpu())
                    for band in range(3):band_counts[band]+=int((band_idx==band).sum().cpu())
                    for cls in range(3):
                        mask=y==cls;class_seen[cls]+=int(mask.sum().cpu())
                        class_harmful[cls]+=int((harmful&mask).sum().cpu())
                        class_drop[cls]+=float(drop.clamp_min(0)[mask].sum().detach().cpu())
                        for band in range(3):class_counts[cls,band]+=int((mask&(band_idx==band)).sum().cpu())
                else:
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        logits=model(x);cls_loss=criterion(logits,y)
                        margin_loss=cls_loss.detach()*0;loss=cls_loss
                if not torch.isfinite(loss):raise RuntimeError(f'{name} epoch={epoch} batch={batch_index}: non-finite loss')
                scaler.scale(loss).backward();scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
                scaler.step(optimizer);scaler.update()
                losses.append(float(loss.detach().cpu()));cls_losses.append(float(cls_loss.detach().cpu()))
                margin_losses.append(float(margin_loss.detach().cpu()))
                if batch_index==1 or batch_index%max(1,len(train_loader)//18)==0 or batch_index==len(train_loader):
                    log(name,f'E{epoch} batch {batch_index}/{len(train_loader)} loss={losses[-1]:.4f} mean={np.mean(losses):.4f}')
            logits,val_y,val_keys,val_codes=eval_spatial(model,val_loader,device,use_amp)
            assert_ordered_keys(val_keys,resources['val_keys'],name+' micro-Val')
            val_scores=binary_score(torch.as_tensor(logits)).cpu().numpy()
            threshold=threshold_from_val(val_y,val_scores)
            val_metrics,_=metrics_celeba(val_y,logits,threshold)
            row={'epoch':epoch,'train_loss':float(np.mean(losses)),'val_acer':val_metrics['acer'],
                'val_auc':val_metrics['auc'],'val_apcer':val_metrics['apcer'],
                'val_bpcer':val_metrics['bpcer'],'val_threshold':threshold,
                'seconds':time.monotonic()-start}
            history.append(row);log(name,f'epoch {epoch} Val {row}')
            if csmr_enabled:
                d={'epoch':epoch,'lambda':CSMR_LAMBDA if csmr_n else 0.,
                    'train_total_loss':float(np.mean(losses)),
                    'train_cls_loss':float(np.mean(cls_losses)),
                    'train_margin_loss':float(np.mean(margin_losses)) if csmr_n else None,
                    'val_acer':val_metrics['acer'],'val_auc':val_metrics['auc'],
                    'val_apcer':val_metrics['apcer'],'val_bpcer':val_metrics['bpcer'],
                    'val_threshold':threshold}
                for band,count in zip(BANDS,band_counts.tolist()):
                    d[f'worst_{band.lower()}_count']=count if csmr_n else None
                    d[f'worst_{band.lower()}_fraction']=count/csmr_n if csmr_n else None
                d.update({'harmful_fraction':harmful_n/csmr_n if csmr_n else None,
                    'mean_harmful_margin_drop':harmful_drop_sum/harmful_n if harmful_n else None,
                    'mean_clean_aligned_margin':clean_margin_sum/csmr_n if csmr_n else None,
                    'mean_worst_aligned_margin':worst_margin_sum/csmr_n if csmr_n else None,
                    'mean_sampled_gain':gain_sum/csmr_n if csmr_n else None,
                    'mean_clip_fraction':clip_sum/csmr_n if csmr_n else None})
                diagnostics.append(d)
                if csmr_n and max(d[f'worst_{b.lower()}_fraction'] for b in BANDS)>.95:
                    log(name,'WARNING: one spectral band selected in >95% of samples')
                if csmr_n and d['mean_clip_fraction']>.05:
                    log(name,'WARNING: clipping >5%; do not tune on LCC')
                for cls,cls_name in enumerate(('Real','Physical','Digital')):
                    n=int(class_seen[cls]);cr={'epoch':epoch,'class':cls_name,
                        'n':n if csmr_n else None,
                        'harmful_fraction':int(class_harmful[cls])/n if n else None,
                        'mean_margin_drop':float(class_drop[cls])/n if n else None}
                    for band in range(3):
                        cr[f'worst_{BANDS[band].lower()}_count']=int(class_counts[cls,band]) if n else None
                        cr[f'worst_{BANDS[band].lower()}_fraction']=int(class_counts[cls,band])/n if n else None
                    class_rows.append(cr)
            if val_metrics['acer']<best_acer:
                best_acer=val_metrics['acer'];best_epoch=epoch
                best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                log(name,f'NEW BEST epoch={epoch} Val ACER={best_acer:.6f}')
            elif epoch>=EARLIEST_EARLY_STOP_EPOCH and epoch-best_epoch>=EARLY_STOP_PATIENCE:
                log(name,'early stop on micro-Val ACER');break
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('CUDA OOM: restart clean; set BATCH_SIZE=64 for BOTH branches only') from exc
    if best_state is None:raise RuntimeError(f'{name}: no Val-selected state')
    model.load_state_dict(best_state);model.to(device)
    logits,val_y,val_keys,val_codes=eval_spatial(model,val_loader,device,use_amp)
    assert_ordered_keys(val_keys,resources['val_keys'],name+' final Val')
    locked=threshold_from_val(val_y,binary_score(torch.as_tensor(logits)).cpu().numpy())
    val_metrics,_=metrics_celeba(val_y,logits,locked)
    folder=csmr_dir if csmr_enabled else vanilla_dir
    best_path=folder/'best.pth';torch.save({'state_dict':best_state,'epoch':best_epoch},best_path)
    pd.DataFrame(history).to_csv(folder/'training_history.csv',index=False)
    if csmr_enabled:
        pd.DataFrame(diagnostics).to_csv(folder/'training_diagnostics.csv',index=False)
        pd.DataFrame(class_rows).to_csv(folder/'class_diagnostics.csv',index=False)
    make_prediction_frame(val_keys,val_y,val_codes,logits,locked).to_csv(folder/'val_predictions.csv',index=False)
    config={'schema':'minifasnetv2_paired_config_v1','name':name,'train_seed':TRAIN_SEED,
        'stage0_manifest':'micro_manifest_v1_seed42.npz','input_size':80,
        'crop_scale':CROP_SCALE,'channel_order':'BGR','range':'[0,1]',
        'augmentation':'bbox jitter .2; horizontal flip .5; brightness/contrast .3',
        'optimizer':{'name':'AdamW','backbone_lr':BACKBONE_LR,'classifier_lr':CLASSIFIER_LR,
            'weight_decay':WEIGHT_DECAY},'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,
        'patience':EARLY_STOP_PATIENCE,'earliest_early_stop_epoch':EARLIEST_EARLY_STOP_EPOCH,
        'label_smoothing':LABEL_SMOOTHING,'grad_clip':GRAD_CLIP,'amp':use_amp,
        'csmr':{'enabled':csmr_enabled,'warmup_epochs':CSMR_WARMUP_EPOCHS,
            'lambda':CSMR_LAMBDA,'band_centers':CSMR_BAND_CENTERS,
            'sigma':CSMR_BAND_SIGMA,'gain_range':CSMR_GAIN_RANGE,
            'preserve_dc':CSMR_PRESERVE_DC} if csmr_enabled else None,
        'stage0_resource_hashes':resources['fingerprints'],
        'initial_state_sha256':branch_initial_sha}
    strict_json_write(folder/'config.json',config)
    meta={'schema':'minifasnetv2_paired_meta_v1','name':name,
        'selected_epoch':best_epoch,'checkpoint_sha256':sha256_file(best_path),
        'initial_state_sha256':branch_initial_sha,
        'locked_val_threshold':locked,'threshold_source':'CelebA frozen micro-Val only',
        'val':val_metrics,'stage0_resource_hashes':resources['fingerprints']}
    strict_json_write(folder/'meta.json',meta)
    model.cpu();torch.cuda.empty_cache()
    log(name,f'FROZEN best epoch={best_epoch}, SHA={meta["checkpoint_sha256"]}, locked threshold={locked:.6f}')
    return {'name':name,'model':model,'folder':folder,'val':val_metrics,
        'locked':locked,'epoch':best_epoch,'sha':meta['checkpoint_sha256'],
        'initial_sha':branch_initial_sha,'diagnostics':diagnostics}

vanilla=run_branch('minifasnetv2_vanilla_seed100',False)
csmr=run_branch('minifasnetv2_csmr_seed100',True)
if vanilla['initial_sha']!=csmr['initial_sha'] or vanilla['initial_sha']!=initial_sha:
    raise RuntimeError('Paired initialization fingerprint mismatch')
init_json=strict_json_load(comparison_dir/'initialization_fingerprint.json')
init_json['vanilla_initial_state_sha256']=vanilla['initial_sha']
init_json['csmr_initial_state_sha256']=csmr['initial_sha']
strict_json_write(comparison_dir/'initialization_fingerprint.json',init_json)
log('PAIRED',f'Both branches start from identical state SHA {initial_sha}')


[2026-09-29T13:58:41+00:00 +58.2s minifasnetv2_vanilla_seed100] Initial SHA 318dbe2a0fa2c80cc36d4f881aa3967b226255af71a45049529daa5430b93adc; trainable params=434560
[2026-09-29T13:58:44+00:00 +61.4s minifasnetv2_vanilla_seed100] epoch 1/12 start


/tmp/ipykernel_58/3408219131.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_58/3408219131.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[2026-09-29T13:58:53+00:00 +70.3s minifasnetv2_vanilla_seed100] E1 batch 1/40 loss=1.1442 mean=1.1442
[2026-09-29T13:58:53+00:00 +70.4s minifasnetv2_vanilla_seed100] E1 batch 2/40 loss=1.1054 mean=1.1248
[2026-09-29T13:58:53+00:00 +70.6s minifasnetv2_vanilla_seed100] E1 batch 4/40 loss=1.0052 mean=1.0808
[2026-09-29T13:58:53+00:00 +70.7s minifasnetv2_vanilla_seed100] E1 batch 6/40 loss=0.8976 mean=1.0250
[2026-09-29T13:58:54+00:00 +70.8s minifasnetv2_vanilla_seed100] E1 batch 8/40 loss=0.8684 mean=0.9865
[2026-09-29T13:58:55+00:00 +72.5s minifasnetv2_vanilla_seed100] E1 batch 10/40 loss=0.7825 mean=0.9486
[2026-09-29T13:58:56+00:00 +72.9s minifasnetv2_vanilla_seed100] E1 batch 12/40 loss=0.7279 mean=0.9178
[2026-09-29T13:58:57+00:00 +74.4s minifasnetv2_vanilla_seed100] E1 batch 14/40 loss=0.8169 mean=0.8994
[2026-09-29T13:58:58+00:00 +75.2s minifasnetv2_vanilla_seed100] E1 batch 16/40 loss=0.6371 mean=0.8723
[2026-09-29T13:58:59+00:00 +76.1s minifasnetv2_vanilla_seed100] E1 batch 18/40

/tmp/ipykernel_58/3683364734.py:354: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


[2026-09-29T13:59:26+00:00 +103.1s minifasnetv2_vanilla_seed100] epoch 1 Val {'epoch': 1, 'train_loss': 0.699074064195156, 'val_acer': 0.4701188952596923, 'val_auc': 0.4923267463978935, 'val_apcer': 0.07680835197613721, 'val_bpcer': 0.8634294385432474, 'val_threshold': -2.11913800239563, 'seconds': 41.70278173600002}
[2026-09-29T13:59:26+00:00 +103.2s minifasnetv2_vanilla_seed100] NEW BEST epoch=1 Val ACER=0.470119
[2026-09-29T13:59:26+00:00 +103.2s minifasnetv2_vanilla_seed100] epoch 2/12 start
[2026-09-29T13:59:27+00:00 +104.0s minifasnetv2_vanilla_seed100] E2 batch 1/40 loss=0.5023 mean=0.5023
[2026-09-29T13:59:27+00:00 +104.1s minifasnetv2_vanilla_seed100] E2 batch 2/40 loss=0.4482 mean=0.4753


...

[2026-09-29T14:03:14+00:00 +331.4s minifasnetv2_csmr_seed100] E6 batch 2/40 loss=0.6267 mean=0.6272
[2026-09-29T14:03:15+00:00 +332.0s minifasnetv2_csmr_seed100] E6 batch 4/40 loss=0.6780 mean=0.6393
[2026-09-29T14:03:15+00:00 +332.6s minifasnetv2_csmr_seed100] E6 batch 6/40 loss=0.6677 mean=0.6485
[2026-09-29T14:03:16+00:00 +333.2s minifasnetv2_csmr_seed100] E6 batch 8/40 loss=0.6597 mean=0.6520
[2026-09-29T14:03:17+00:00 +333.8s minifasnetv2_csmr_seed100] E6 batch 10/40 loss=0.6276 mean=0.6473
[2026-09-29T14:03:17+00:00 +334.5s minifasnetv2_csmr_seed100] E6 batch 12/40 loss=0.6542 mean=0.6487
[2026-09-29T14:03:18+00:00 +335.1s minifasnetv2_csmr_seed100] E6 batch 14/40 loss=0.6429 mean=0.6511
[2026-09-29T14:03:18+00:00 +335.7s minifasnetv2_csmr_seed100] E6 batch 16/40 loss=0.6431 mean=0.6504
[2026-09-29T14:03:19+00:00 +336.3s minifasnetv2_csmr_seed100] E6 batch 18/40 loss=0.6897 mean=0.6582
[2026-09-29T14:03:20+00:00 +336.9s minifasnetv2_csmr_seed100] E6 batch 20/40 loss=0.6527 mean=0

/tmp/ipykernel_58/3683364734.py:354: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


[2026-09-29T14:03:29+00:00 +346.1s minifasnetv2_csmr_seed100] epoch 6 Val {'epoch': 6, 'train_loss': 0.664711645245552, 'val_acer': 0.024777672540705814, 'val_auc': 0.9952779107385946, 'val_apcer': 0.02982848620432513, 'val_bpcer': 0.019726858877086494, 'val_threshold': 0.4349032938480377, 'seconds': 16.227985856000032}
[2026-09-29T14:03:29+00:00 +346.1s minifasnetv2_csmr_seed100] early stop on micro-Val ACER


/tmp/ipykernel_58/3683364734.py:354: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


[2026-09-29T14:03:33+00:00 +350.5s minifasnetv2_csmr_seed100] FROZEN best epoch=3, SHA=d2b7f39cc22b96f0654f87cd1fbdbb6f3a8042d2944daffb3f0a7de64815f79d, locked threshold=-2.236379
[2026-09-29T14:03:33+00:00 +350.6s PAIRED] Both branches start from identical state SHA 318dbe2a0fa2c80cc36d4f881aa3967b226255af71a45049529daa5430b93adc


In [12]:
lcc_ds=MiniLCCDataset(resources['lcc_df'],lcc_root,resources['lcc_cache'])
lcc_loader=DataLoader(lcc_ds,batch_size=BATCH_SIZE,shuffle=False,drop_last=False,
    num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=NUM_WORKERS>0)
for result in (vanilla,csmr):
    name=result['name'];model=result['model'].to(device)
    log(name,'LCC external-development evaluation begins after both checkpoints are frozen')
    logits,y,keys,frame=eval_lcc_spatial(model,lcc_loader,device,use_amp,
        resources['lcc_df'],result['locked'])
    scores=binary_score(torch.as_tensor(logits)).cpu().numpy()
    metrics=metrics_binary(y,scores,result['locked'],
        coverage=len(y)/len(resources['lcc_df']),candidate_n=len(resources['lcc_df']))
    frame['locked_threshold']=result['locked']
    frame.to_csv(result['folder']/'lcc_predictions.csv',index=False)
    result['lcc']=metrics;model.cpu();torch.cuda.empty_cache()
    log(name,f'LCC HTER={metrics["hter"]:.6f} AUC={metrics["auc"]:.6f} coverage={metrics["detector_coverage"]:.5f}')

delta={'val_acer':csmr['val']['acer']-vanilla['val']['acer'],
    'val_auc':csmr['val']['auc']-vanilla['val']['auc'],
    'lcc_apcer':csmr['lcc']['apcer']-vanilla['lcc']['apcer'],
    'lcc_bpcer':csmr['lcc']['bpcer']-vanilla['lcc']['bpcer'],
    'lcc_hter':csmr['lcc']['hter']-vanilla['lcc']['hter'],
    'lcc_auc':csmr['lcc']['auc']-vanilla['lcc']['auc']}
if delta['lcc_auc']>=.015 and delta['val_acer']<=.010:
    verdict='TRANSFER_STRONG'
elif ((delta['lcc_auc']>0 and delta['val_acer']<=.010) or
      (delta['lcc_hter']<=-.015 and delta['lcc_auc']>=0 and delta['val_acer']<=.010)):
    verdict='TRANSFER_POSITIVE'
else:
    verdict='TRANSFER_NO_GAIN'
paired_rows=[]
for label,split,metric in [('Val ACER','val','acer'),('Val AUC','val','auc'),
    ('LCC APCER','lcc','apcer'),('LCC BPCER','lcc','bpcer'),
    ('LCC HTER','lcc','hter'),('LCC AUC','lcc','auc')]:
    paired_rows.append({'metric':label,'vanilla':vanilla[split][metric],
        'csmr':csmr[split][metric],
        'delta_csmr_minus_vanilla':csmr[split][metric]-vanilla[split][metric]})
pd.DataFrame(paired_rows).to_csv(comparison_dir/'paired_metrics.csv',index=False)
print('| Metric | MiniFASNet Vanilla | MiniFASNet + CSMR | Delta |\n|---|---:|---:|---:|')
for row in paired_rows:print(f'| {row["metric"]} | {row["vanilla"]:.6f} | {row["csmr"]:.6f} | {row["delta_csmr_minus_vanilla"]:+.6f} |')
log('TRANSFER',f'{verdict}; fixed development descriptor, no tuning')
# Persist development-only verdict before reading optional already-opened Test-2000.
summary={'schema':'minifasnetv2_csmr_backbone_transfer_seed100_v1',
    'role':'backbone-transfer external-development screen',
    'training_seed':TRAIN_SEED,'frozen_stage0_manifest':'micro_manifest_v1_seed42.npz',
    'resource_hashes':resources['fingerprints'],
    'architecture_source_sha256':loading_audit['architecture_source_sha256'],
    'pretrained_checkpoint_sha256':loading_audit['checkpoint_sha256'],
    'initial_state_sha256':initial_sha,
    'preprocessing':{'input_size':[80,80],'crop_scale':2.7,'channel_order':'BGR',
        'tensor_range':'[0,1]','normalization':'none','face_localization':'frozen Stage0 bbox_xyxy'},
    'vanilla':{'selected_epoch':vanilla['epoch'],'locked_val_threshold':vanilla['locked'],
        'checkpoint_sha256':vanilla['sha'],'val':vanilla['val'],'lcc':vanilla['lcc']},
    'csmr':{'selected_epoch':csmr['epoch'],'locked_val_threshold':csmr['locked'],
        'checkpoint_sha256':csmr['sha'],'val':csmr['val'],'lcc':csmr['lcc']},
    'delta_csmr_minus_vanilla':delta,
    'transfer_rule':{'strong':'delta_lcc_auc >= +0.015 AND delta_val_acer <= +0.010',
        'positive':'delta_lcc_auc > 0 AND delta_val_acer <= +0.010 OR delta_lcc_hter <= -0.015 AND delta_lcc_auc >= 0 AND delta_val_acer <= +0.010'},
    'transfer_verdict':verdict,
    'fixed_opened_test2000':None}
strict_json_write(comparison_dir/'transfer_summary.json',summary)
log('TRANSFER','Development summary saved before optional Test access')

report=['# MiniFASNetV2 backbone-transfer benchmark','',
    '## Objective','Does frozen CSMR-v1 add value to a paired PAD-pretrained MiniFASNetV2 benchmark?','',
    '## Provenance and loading',
    f'Source SHA `{loading_audit["architecture_source_sha256"]}`; official checkpoint SHA `{loading_audit["checkpoint_sha256"]}`. Loaded {loading_audit["loaded_pretrained_key_count"]} compatible pretrained keys; skipped `prob.weight` only and initialized a fresh 3-class classifier.','',
    '## Input and paired initialization',
    f'Stage-0 face bbox coordinates, official 2.7 crop geometry, direct 80×80 BGR, float [0,1], no MNV3 normalization. Both runs began at SHA `{initial_sha}`. Train/Val remain frozen 5K/2K from seed42 manifest; training RNG seed 100.','',
    '## Vanilla result',f'Best epoch {vanilla["epoch"]}; locked Val threshold {vanilla["locked"]:.6f}.','',
    '## CSMR result',f'Best epoch {csmr["epoch"]}; locked Val threshold {csmr["locked"]:.6f}.','',
    '## Val/LCC paired comparison','| Metric | Vanilla | CSMR | Delta (CSMR - Vanilla) |',
    '|---|---:|---:|---:|']
for row in paired_rows:report.append(f'| {row["metric"]} | {row["vanilla"]:.6f} | {row["csmr"]:.6f} | {row["delta_csmr_minus_vanilla"]:+.6f} |')
report+=['','## CSMR diagnostics']
if csmr['diagnostics']:
    d=csmr['diagnostics'][-1]
    report.append('Final epoch LOW/MID/HIGH worst-view fractions: '+
        '/'.join(f'{d[f"worst_{b.lower()}_fraction"]:.3f}' for b in BANDS)+
        f'; harmful fraction {d["harmful_fraction"]:.3f}; mean harmful margin drop {d["mean_harmful_margin_drop"]}; mean gain {d["mean_sampled_gain"]:.3f}; mean clip fraction {d["mean_clip_fraction"]:.4f}.')
report+=['','## Transfer verdict',f'**{verdict}** — pre-declared development descriptor, not statistical significance. No CSMR parameter is adjusted from LCC.','',
    '## Test-2000 recovery','Pending: recover the exact previously opened subset only after this development summary is saved.','',
    '## Interpretation and limitations',
    'This experiment isolates the incremental effect of frozen CSMR-v1 within MiniFASNetV2. It does not isolate MiniFASNet architecture from its PAD-specific pretrained initialization. MNV3-S used ImageNet pretraining, so absolute backbone metrics also reflect crop/input and pretraining differences. One paired seed and LCC external-development evidence do not establish architecture-independent generalization.','',
    '## Recommended next experiment',
    ('Replicate the fixed paired comparison with independent seeds and the same protocol.' if verdict!='TRANSFER_NO_GAIN' else 'Record the no-gain result; do not tune frozen spectral parameters on LCC.'),'']
(comparison_dir/'transfer_report.md').write_text('\n'.join(report))
log('OUTPUT',f'Development summary/report saved before Test metadata access: {comparison_dir}')


[2026-09-29T14:03:34+00:00 +350.7s minifasnetv2_vanilla_seed100] LCC external-development evaluation begins after both checkpoints are frozen


/tmp/ipykernel_58/3683364734.py:375: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


[2026-09-29T14:03:58+00:00 +375.6s minifasnetv2_vanilla_seed100] LCC HTER=0.318242 AUC=0.819484 coverage=0.99894
[2026-09-29T14:03:58+00:00 +375.6s minifasnetv2_csmr_seed100] LCC external-development evaluation begins after both checkpoints are frozen


/tmp/ipykernel_58/3683364734.py:405: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ True  True  True ... False False False]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  full.loc[valid, "pred_real"] = full.loc[valid, "pad_logit_score"] >= float(threshold)
/tmp/ipykernel_58/3683364734.py:375: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


[2026-09-29T14:04:15+00:00 +392.2s minifasnetv2_csmr_seed100] LCC HTER=0.287548 AUC=0.846264 coverage=0.99894
| Metric | MiniFASNet Vanilla | MiniFASNet + CSMR | Delta |
|---|---:|---:|---:|
| Val ACER | 0.016909 | 0.021691 | +0.004782 |
| Val AUC | 0.998290 | 0.996634 | -0.001657 |
| LCC APCER | 0.518233 | 0.495405 | -0.022828 |
| LCC BPCER | 0.118252 | 0.079692 | -0.038560 |
| LCC HTER | 0.318242 | 0.287548 | -0.030694 |
| LCC AUC | 0.819484 | 0.846264 | +0.026779 |
[2026-09-29T14:04:15+00:00 +392.2s TRANSFER] TRANSFER_STRONG; fixed development descriptor, no tuning
[2026-09-29T14:04:15+00:00 +392.2s TRANSFER] Development summary saved before optional Test access
[2026-09-29T14:04:15+00:00 +392.2s OUTPUT] Development summary/report saved before Test metadata access: /kaggle/working/minifasnetv2_csmr_seed100/comparison


/tmp/ipykernel_58/3683364734.py:405: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ True  True  True ... False  True False]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  full.loc[valid, "pred_real"] = full.loc[valid, "pad_logit_score"] >= float(threshold)


In [13]:
# Verbatim E1 v5.3 generator functions from e1_v5_3_training.ipynb.
# Defined before development; executed ONLY after a persisted PASS freeze gate.
SCRFD_BBOX_EXPANSION_FACTOR = 1.55
CELEBA_FALLBACK_EXPANSION_FACTOR = 1.50
TRAIN_MIN_FACE_SIZE = 48
SCRFD_PRIMARY_INPUT_SIZE = (640, 640)
SCRFD_FALLBACK_INPUT_SIZES = [(480, 480), (320, 320)]
SCRFD_CONF_THRESH = 0.50
RELIABLE_RAW_IOU_MIN = 0.50
RELIABLE_CENTER_SHIFT_MAX = 0.20
DATA_ROOT = CELEBA_ROOT
E1_GENERATOR_NOTEBOOK_SHA256 = "971d4bbf0ff709ab073b0a7266f26132cf7e23cc05ad73bf08f86d3ff509d539"

def bbox_wh(b):
    x1, y1, x2, y2 = map(float, b)
    return np.array([x2 - x1, y2 - y1], dtype=np.float64)


def bbox_area(b):
    w, h = bbox_wh(b)
    return max(0.0, float(w)) * max(0.0, float(h))


def bbox_center(b):
    x1, y1, x2, y2 = map(float, b)
    return np.array([(x1 + x2) / 2.0, (y1 + y2) / 2.0], dtype=np.float64)


def bbox_intersection_area(a, b):
    ax1, ay1, ax2, ay2 = map(float, a)
    bx1, by1, bx2, by2 = map(float, b)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    return max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)


def bbox_iou(a, b):
    inter = bbox_intersection_area(a, b)
    union = bbox_area(a) + bbox_area(b) - inter
    return 0.0 if union <= 0 else float(inter / union)


def normalized_center_shift(reference, candidate):
    ref_wh = bbox_wh(reference)
    denom = max(float(ref_wh[0]), float(ref_wh[1]), 1e-8)
    return float(np.linalg.norm(bbox_center(reference) - bbox_center(candidate)) / denom)


def read_celeba_bbox_xyxy(img_path, image_shape=None):
    """Convert CelebA-Spoof 224-reference *_BB.txt coordinates to native-image xyxy."""
    if image_shape is None:
        image = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f"Cannot read image: {img_path}")
        h_img, w_img = image.shape[:2]
    else:
        h_img, w_img = image_shape[:2]

    bb_path = os.path.splitext(img_path)[0] + "_BB.txt"
    if not os.path.exists(bb_path):
        raise FileNotFoundError(f"Missing bbox file: {bb_path}")

    with open(bb_path, "r") as f:
        parts = f.readline().strip().split()
    if len(parts) < 4:
        raise ValueError(f"bbox needs >=4 values, got {parts}")

    x, y, w, h = map(float, parts[:4])
    x = x / 224.0 * w_img
    y = y / 224.0 * h_img
    w = w / 224.0 * w_img
    h = h / 224.0 * h_img
    if w <= 0 or h <= 0:
        raise ValueError(f"Non-positive CelebA bbox at {bb_path}: {(x, y, w, h)}")
    return np.array([x, y, x + w, y + h], dtype=np.float64)


def _select_target_detection(bboxes, gt_bbox):
    if bboxes is None or len(bboxes) == 0:
        return None

    candidates = []
    for row in bboxes:
        det = np.asarray(row[:4], dtype=np.float64)
        conf = float(row[4])
        candidates.append({
            "bbox": det,
            "confidence": conf,
            "raw_iou": bbox_iou(gt_bbox, det),
            "center_shift": normalized_center_shift(gt_bbox, det),
            "n_faces": int(len(bboxes)),
        })

    return max(candidates, key=lambda x: (x["raw_iou"], x["confidence"]))


def _scrfd_detect(detector, image_bgr, input_size):
    """
    Prefer detect(..., input_size=...). Older InsightFace variants are handled
    by temporarily changing detector.input_size.
    """
    try:
        return detector.detect(
            image_bgr,
            input_size=tuple(input_size),
            max_num=0,
            metric="default",
        )
    except TypeError:
        old_input_size = getattr(detector, "input_size", None)
        detector.input_size = tuple(input_size)
        try:
            return detector.detect(image_bgr, max_num=0, metric="default")
        finally:
            detector.input_size = old_input_size


def build_cache_record(detector, rel_path):
    img_path = os.path.join(DATA_ROOT, rel_path)
    image = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if image is None:
        return {"status": "INVALID", "reason": "image_read_failure"}

    try:
        gt = read_celeba_bbox_xyxy(img_path, image.shape)
    except Exception as exc:
        return {
            "status": "INVALID",
            "reason": f"celeba_bbox_error:{type(exc).__name__}",
        }

    gt_w, gt_h = bbox_wh(gt)
    gt_min_side = float(min(gt_w, gt_h))

    sizes_to_try = [SCRFD_PRIMARY_INPUT_SIZE] + list(SCRFD_FALLBACK_INPUT_SIZES)
    best_unreliable = None

    for input_size in sizes_to_try:
        bboxes, _ = _scrfd_detect(detector, image, input_size=input_size)
        candidate = _select_target_detection(bboxes, gt)
        if candidate is None:
            continue

        candidate["input_size"] = list(input_size)

        if (
            candidate["raw_iou"] >= RELIABLE_RAW_IOU_MIN
            and candidate["center_shift"] <= RELIABLE_CENTER_SHIFT_MAX
        ):
            det = candidate["bbox"]
            dw, dh = bbox_wh(det)
            return {
                "status": "VALID",
                "bbox_source": f"SCRFD_{input_size[0]}",
                "bbox_xyxy": [float(x) for x in det],
                "crop_factor": SCRFD_BBOX_EXPANSION_FACTOR,
                "face_min_side": float(min(dw, dh)),
                "confidence": candidate["confidence"],
                "n_faces": candidate["n_faces"],
                "raw_iou_to_celeba": candidate["raw_iou"],
                "center_shift_norm": candidate["center_shift"],
                "det_input_size": list(input_size),
                "gt_min_side": gt_min_side,
            }

        if best_unreliable is None or (
            candidate["raw_iou"], candidate["confidence"]
        ) > (
            best_unreliable["raw_iou"], best_unreliable["confidence"]
        ):
            best_unreliable = candidate

    # Do not auto-drop detector failures: the diagnostic showed class-dependent
    # failure, especially for Physical Spoof. Keep sample through CelebA fallback.
    if best_unreliable is None:
        return {
            "status": "FALLBACK_DET_FAIL",
            "bbox_source": "CELEBA_FALLBACK_DET_FAIL",
            "bbox_xyxy": [float(x) for x in gt],
            "crop_factor": CELEBA_FALLBACK_EXPANSION_FACTOR,
            "face_min_side": gt_min_side,
            "confidence": None,
            "n_faces": 0,
            "raw_iou_to_celeba": None,
            "center_shift_norm": None,
            "det_input_size": None,
            "gt_min_side": gt_min_side,
        }

    # Suspicious geometry: retain Val/Test by GT fallback, but Train filters it.
    return {
        "status": "FALLBACK_SUSPICIOUS",
        "bbox_source": "CELEBA_FALLBACK_SUSPICIOUS",
        "bbox_xyxy": [float(x) for x in gt],
        "crop_factor": CELEBA_FALLBACK_EXPANSION_FACTOR,
        "face_min_side": gt_min_side,
        "confidence": best_unreliable["confidence"],
        "n_faces": best_unreliable["n_faces"],
        "raw_iou_to_celeba": best_unreliable["raw_iou"],
        "center_shift_norm": best_unreliable["center_shift"],
        "det_input_size": best_unreliable["input_size"],
        "gt_min_side": gt_min_side,
    }


In [15]:
%pip install -q "insightface==2.0" "onnxruntime"

import insightface
import onnxruntime as ort

print("InsightFace:", insightface.__version__)
print("ONNX Runtime providers:", ort.get_available_providers())

assert insightface.__version__ == source_policy["insightface_version"]
assert "CPUExecutionProvider" in ort.get_available_providers()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 13.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 71.7 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.
InsightFace: 2.0
ONNX Runtime providers: ['AzureExecutionProvider', 'CPUExecutionProvider']


In [16]:
# Recovery of the PREVIOUSLY OPENED Test-2000 identities, after persisted Val/LCC results.
from sklearn.model_selection import train_test_split
from datetime import datetime, timezone
TEST_SUBSET_N=2000
TEST_SUBSET_SEED=4242
PREVIOUS_OPENED_ORDERED_SHA256='df3416273c3e7d8bf0caa3b010454ee73236211e034ec12bfb481f056fe237b8'
PREVIOUS_OPENED_CACHE_SHA256='7c4e5b1d326fbf7781c7d049923d557f0e0c298837e32d9ffc614136ecbe0d68'
# These hashes and the historical PASS are embedded in the executed csmr-seed100.ipynb.
if not (comparison_dir/'transfer_summary.json').is_file() or not (comparison_dir/'transfer_report.md').is_file():
    raise RuntimeError('Val/LCC development decision is not persisted; refusing Test access')
persisted=strict_json_load(comparison_dir/'transfer_summary.json')
if (persisted['transfer_verdict']!=verdict or
    persisted['resource_hashes']!=resources['fingerprints'] or
    persisted['initial_state_sha256']!=initial_sha):
    raise RuntimeError('Persisted development summary changed; refusing Test access')
for result in (vanilla,csmr):
    if sha256_file(result['folder']/'best.pth')!=result['sha']:
        raise RuntimeError(f'{result["name"]} selected checkpoint changed before Test')
metadata_path=celeba_root/'metas/intra_test/test_label.json'
if not metadata_path.is_file():
    raise FileNotFoundError(f'Official Test metadata missing: {metadata_path}')
# The original seed100 notebook recorded FREEZE CHECK=PASS and already opened Test.
# This marker records a recovery of its fixed identities, never a new independent confirmation.
strict_json_write(comparison_dir/'celeba_test2000_recovery_marker.json',{
    'schema':'minifasnet_test2000_recovery_marker_v1',
    'reopened_at_utc':datetime.now(timezone.utc).isoformat(),
    'development_summary_sha256':sha256_file(comparison_dir/'transfer_summary.json'),
    'historical_freeze_check':'PASS',
    'historical_test_state':'already opened in completed csmr-seed100.ipynb',
    'expected_ordered_path_fingerprint_sha256':PREVIOUS_OPENED_ORDERED_SHA256,
    'metadata_path':str(metadata_path),'subset_seed':TEST_SUBSET_SEED,'subset_n':TEST_SUBSET_N})
log('TEST','Recovery marker saved; now reading official metadata to reconstruct exact opened identities')
meta=strict_json_load(metadata_path)
if not isinstance(meta,dict) or len(meta)<TEST_SUBSET_N:
    raise RuntimeError('Official Test metadata malformed or too small')
official_keys=sorted(map(str,meta))
if len(official_keys)!=len(set(official_keys)) or not all(k.startswith('Data/test/') for k in official_keys):
    raise RuntimeError('Official Test keys are not the expected Data/test split')
def official_code(key):
    row=meta[key]
    if not isinstance(row,(list,tuple)) or len(row)<=40:
        raise RuntimeError(f'Official Test attack code at index 40 missing: {key}')
    value=row[40]
    if isinstance(value,bool) or int(value)!=value or int(value)<0:
        raise RuntimeError(f'Invalid official Test attack code: {key}')
    return int(value)
official_codes=np.asarray([official_code(k) for k in official_keys],dtype=np.int64)
official_labels=labels_from_attack_codes(official_codes)
if set(map(int,official_labels))!={0,1,2}:
    raise RuntimeError('Official Test metadata does not contain all three PAD classes')
if len(official_keys)==TEST_SUBSET_N:
    selected_keys=list(official_keys)
else:
    selected_keys,_=train_test_split(official_keys,train_size=TEST_SUBSET_N,
        stratify=official_labels,random_state=TEST_SUBSET_SEED)
if (len(selected_keys)!=TEST_SUBSET_N or len(set(selected_keys))!=TEST_SUBSET_N or
    ordered_path_fingerprint(selected_keys)!=PREVIOUS_OPENED_ORDERED_SHA256):
    raise RuntimeError('Reconstructed Test-2000 differs from the previously opened subset; STOP')
if set(selected_keys)&(set(map(str,resources['train_keys']))|set(map(str,resources['val_keys']))):
    raise RuntimeError('Recovered Test keys overlap frozen Train/Val')
selected_codes=np.asarray([official_code(k) for k in selected_keys],dtype=np.int64)
selected_labels=labels_from_attack_codes(selected_codes)
if Counter(map(int,selected_labels))!=Counter({0:593,1:1057,2:350}):
    raise RuntimeError('Recovered Test-2000 class counts differ from completed seed100 notebook')
manifest=pd.DataFrame({'sample_key':selected_keys,
    'binary_label_real':(selected_labels==0).astype(np.int64),
    'three_class_label':selected_labels,'attack_code':selected_codes,
    'official_split':'test','sampling_seed':TEST_SUBSET_SEED})
test_manifest_path=comparison_dir/'celeba_test2000_manifest.csv'
manifest.to_csv(test_manifest_path,index=False)
manifest_sha=ordered_path_fingerprint(selected_keys)
metadata_sha=sha256_file(metadata_path)
log('TEST',f'Recovered exact {len(selected_keys)} opened Test keys; ordered SHA={manifest_sha}; classes={dict(Counter(map(int,selected_labels)))}')

# Rebuild face localization with the verbatim E1 v5.3 generator, independent of Mini's 2.7 crop.
stage0_bbox_payload=strict_json_load(resources['stage0_dir']/'celeba_micro_bbox_cache.json')
source_policy=stage0_bbox_payload.get('policy',{})
fixed_policy={'detector':'SCRFD-500M / buffalo_s detection',
    'detector_model':'det_500m.onnx','primary_input_size':[640,640],
    'fallback_input_sizes':[[480,480],[320,320]],'det_thresh':.50,
    'reliable_raw_iou_min':.50,'reliable_center_shift_max':.20,
    'scrfd_crop_factor':1.55,'celeba_fallback_crop_factor':1.50,
    'train_min_face_px':48}
for field,expected in fixed_policy.items():
    if source_policy.get(field)!=expected:
        raise RuntimeError(f'Stage0/E1 detector policy mismatch: {field}: {source_policy.get(field)}')
detector_source_path=Path(SCRFD_MODEL_PATH).expanduser()
if not detector_source_path.is_file():
    raise FileNotFoundError(f'SCRFD dataset file missing: {detector_source_path}')
detector_sha=sha256_file(detector_source_path)
if detector_sha!=resources['fingerprints']['lcc_detector_sha256']:
    raise RuntimeError('SCRFD dataset model SHA differs from frozen Stage0')
detector_path=Path(SCRFD_MODEL_ROOT).expanduser()/'models/buffalo_s/det_500m.onnx'
detector_path.parent.mkdir(parents=True,exist_ok=True)
shutil.copy2(detector_source_path,detector_path)
if sha256_file(detector_path)!=detector_sha:
    raise RuntimeError('SCRFD model copy SHA verification failed')
try:
    import insightface
    import onnxruntime as ort
    from insightface.app import FaceAnalysis
except ImportError as exc:
    raise RuntimeError('Install InsightFace==2.0 and ONNX Runtime in a clean Kaggle session; exact E1 bbox generation cannot continue') from exc
insightface_version=getattr(insightface,'__version__','unknown')
if insightface_version!=source_policy.get('insightface_version'):
    raise RuntimeError(f'InsightFace version differs from Stage0: {insightface_version} vs {source_policy.get("insightface_version")}')
providers=ort.get_available_providers()
if 'CUDAExecutionProvider' in providers:
    scrfd_providers=['CUDAExecutionProvider','CPUExecutionProvider'];scrfd_ctx_id=0
elif 'CPUExecutionProvider' in providers:
    scrfd_providers=['CPUExecutionProvider'];scrfd_ctx_id=-1
else:
    raise RuntimeError(f'No compatible ONNX provider: {providers}')
face_app=FaceAnalysis(name='buffalo_s',root=str(Path(SCRFD_MODEL_ROOT).expanduser()),
    allowed_modules=['detection'],providers=scrfd_providers)
face_app.prepare(ctx_id=scrfd_ctx_id,det_size=SCRFD_PRIMARY_INPUT_SIZE,det_thresh=SCRFD_CONF_THRESH)
detector=face_app.models.get('detection')
if detector is None:raise RuntimeError('SCRFD detector absent from buffalo_s')
records={}
for index,key in enumerate(selected_keys,1):
    records[key]=build_cache_record(detector,key)
    if index==1 or index%100==0 or index==TEST_SUBSET_N:
        log('TEST',f'SCRFD bbox {index}/{TEST_SUBSET_N}; status={records[key].get("status")}')
del detector,face_app
torch.cuda.empty_cache()
if set(records.keys())!=set(manifest.set_index('sample_key').index.astype(str)) or len(records)!=TEST_SUBSET_N:
    raise RuntimeError('Test bbox cache keys differ from exact fixed Test-2000 manifest')
status_counts=Counter(str(rec.get('status','')) for rec in records.values())
invalid_n=sum(v for k,v in status_counts.items() if k not in CELEBA_BBOX_STATUS_FACTORS)
cache_payload={'schema':'celeba_test2000_bbox_cache_v1','policy':source_policy,
    'records':records,'selected_n':TEST_SUBSET_N,
    'ordered_path_fingerprint_sha256':manifest_sha,
    'official_test_metadata_sha256':metadata_sha,'detector_sha256':detector_sha,
    'insightface_version':insightface_version,'onnxruntime_version':ort.__version__,
    'onnx_providers':scrfd_providers,
    'e1_generator_notebook_sha256':E1_GENERATOR_NOTEBOOK_SHA256,
    'source_stage0_bbox_cache_sha256':resources['fingerprints']['celeba_bbox_cache_sha256'],
    'status_counts':dict(status_counts),'valid_n':int(status_counts.get('VALID',0)),
    'fallback_det_fail_n':int(status_counts.get('FALLBACK_DET_FAIL',0)),
    'fallback_suspicious_n':int(status_counts.get('FALLBACK_SUSPICIOUS',0)),
    'invalid_failure_n':int(invalid_n)}
cache_path=comparison_dir/'celeba_test2000_bbox_cache.json'
strict_json_write(cache_path,cache_payload)
cache_sha=sha256_file(cache_path)
log('TEST',f'Saved exact-key E1 bbox cache SHA={cache_sha}; statuses={dict(status_counts)}')
if invalid_n:
    failed=[(k,v.get('reason',v.get('status'))) for k,v in records.items()
        if v.get('status') not in CELEBA_BBOX_STATUS_FACTORS]
    raise RuntimeError(f'{invalid_n} INVALID/failure records; first={failed[:3]}; no resampling')
for key in selected_keys:
    validate_bbox_record(records[key],'celeba',key)
    if not (celeba_root/key).is_file():raise FileNotFoundError(celeba_root/key)

# Both branches consume the same fixed identities and the same newly reconstructed face cache.
test_ds=MiniCelebADataset(selected_keys,selected_labels,selected_codes,celeba_root,records,train=False)
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,drop_last=False,
    num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=NUM_WORKERS>0)
for result,filename in ((vanilla,'vanilla_test_predictions.csv'),(csmr,'csmr_test_predictions.csv')):
    if sha256_file(result['folder']/'best.pth')!=result['sha']:
        raise RuntimeError(f'{result["name"]} checkpoint SHA changed before Test')
    model=result['model'].to(device)
    logits,y,keys,output_codes=eval_spatial(model,test_loader,device,use_amp)
    assert_ordered_keys(keys,selected_keys,result['name']+' fixed opened Test')
    if not np.array_equal(y,selected_labels) or not np.array_equal(output_codes,selected_codes):
        raise RuntimeError('Recovered Test labels/order changed')
    metrics,_=metrics_celeba(y,logits,result['locked'])
    if metrics['n_scored']!=TEST_SUBSET_N or metrics['detector_coverage']!=1.:
        raise RuntimeError('Test coverage incomplete; no resampling')
    make_prediction_frame(keys,y,output_codes,logits,result['locked']).to_csv(comparison_dir/filename,index=False)
    result['test']=metrics;model.cpu();torch.cuda.empty_cache()
    log(result['name'],f'Test ACER={metrics["acer"]:.6f} AUC={metrics["auc"]:.6f}')
rows=[]
for metric in ('apcer','bpcer','acer','auc'):
    rows.append({'metric':'Test '+metric.upper(),'vanilla':vanilla['test'][metric],
        'csmr':csmr['test'][metric],
        'delta_csmr_minus_vanilla':csmr['test'][metric]-vanilla['test'][metric]})
pd.DataFrame(rows).to_csv(comparison_dir/'test2000_metrics.csv',index=False)
summary['fixed_opened_test2000']={
    'role':'reconstructed fixed opened descriptive benchmark, not pristine confirmation',
    'manifest_sha256':sha256_file(test_manifest_path),'bbox_cache_sha256':cache_sha,
    'historical_bbox_cache_sha256':PREVIOUS_OPENED_CACHE_SHA256,
    'bbox_cache_status_counts':dict(status_counts),'invalid_failure_n':invalid_n,
    'detector_sha256':detector_sha,'insightface_version':insightface_version,
    'onnxruntime_version':ort.__version__,'providers':scrfd_providers,
    'e1_generator_notebook_sha256':E1_GENERATOR_NOTEBOOK_SHA256,
    'official_metadata_sha256':metadata_sha,
    'ordered_path_fingerprint_sha256':manifest_sha,'test_subset_seed':TEST_SUBSET_SEED,
    'n':TEST_SUBSET_N,'vanilla':vanilla['test'],'csmr':csmr['test'],
    'delta_csmr_minus_vanilla':{m:csmr['test'][m]-vanilla['test'][m]
        for m in ('apcer','bpcer','acer','auc')}}
strict_json_write(comparison_dir/'transfer_summary.json',summary)
report_start=report.index('## Test-2000 recovery')
report[report_start+1]=('Recovered the exact previously opened 2,000 keys using the original seed 4242 '
    f'and three-class stratification; ordered SHA `{manifest_sha}`. Rebuilt E1 SCRFD bbox cache '
    f'SHA `{cache_sha}` with {dict(status_counts)}; detector SHA `{detector_sha}`. '
    'This is a fixed opened descriptive benchmark, not a new independent confirmation. '
    'Both models use their locked Val thresholds and MiniFASNet 2.7 crop.')
report[report_start+2:report_start+2]=['','| Metric | Vanilla | CSMR | Delta (CSMR - Vanilla) |',
    '|---|---:|---:|---:|']+[f'| {r["metric"]} | {r["vanilla"]:.6f} | {r["csmr"]:.6f} | {r["delta_csmr_minus_vanilla"]:+.6f} |' for r in rows]
(comparison_dir/'transfer_report.md').write_text('\n'.join(report))
log('TEST','Recovered fixed opened Test-2000, shared bbox cache, both predictions and descriptive metrics saved')


[2026-09-29T14:05:41+00:00 +478.3s TEST] Recovery marker saved; now reading official metadata to reconstruct exact opened identities
[2026-09-29T14:05:42+00:00 +478.9s TEST] Recovered exact 2000 opened Test keys; ordered SHA=df3416273c3e7d8bf0caa3b010454ee73236211e034ec12bfb481f056fe237b8; classes={1: 1057, 0: 593, 2: 350}
[2026-09-29T14:05:42+00:00 +479.2s TEST] SCRFD bbox 1/2000; status=VALID
[2026-09-29T14:05:47+00:00 +484.5s TEST] SCRFD bbox 100/2000; status=VALID
[2026-09-29T14:05:52+00:00 +489.4s TEST] SCRFD bbox 200/2000; status=VALID
[2026-09-29T14:05:57+00:00 +494.5s TEST] SCRFD bbox 300/2000; status=VALID
[2026-09-29T14:06:03+00:00 +500.4s TEST] SCRFD bbox 400/2000; status=VALID
[2026-09-29T14:06:08+00:00 +505.4s TEST] SCRFD bbox 500/2000; status=VALID
[2026-09-29T14:06:13+00:00 +510.4s TEST] SCRFD bbox 600/2000; status=VALID
[2026-09-29T14:06:19+00:00 +516.0s TEST] SCRFD bbox 700/2000; status=VALID
[2026-09-29T14:06:24+00:00 +521.2s TEST] SCRFD bbox 800/2000; status=VALID
[2

/tmp/ipykernel_58/3683364734.py:354: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


[2026-09-29T14:07:30+00:00 +586.9s minifasnetv2_vanilla_seed100] Test ACER=0.196058 AUC=0.937166


/tmp/ipykernel_58/3683364734.py:354: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


[2026-09-29T14:07:37+00:00 +593.8s minifasnetv2_csmr_seed100] Test ACER=0.213520 AUC=0.935649
[2026-09-29T14:07:37+00:00 +593.8s TEST] Recovered fixed opened Test-2000, shared bbox cache, both predictions and descriptive metrics saved


In [17]:
from IPython.display import FileLink,display
ZIP_PATH=Path('/kaggle/working/minifasnetv2_csmr_seed100.zip')
if ZIP_PATH.exists():raise RuntimeError(f'ZIP already exists: {ZIP_PATH}')
write_zip(output_root,ZIP_PATH)
print(f'ZIP: {ZIP_PATH} | {ZIP_PATH.stat().st_size/2**20:.2f} MB')
display(FileLink(str(ZIP_PATH)))


ZIP: /kaggle/working/minifasnetv2_csmr_seed100.zip | 5.65 MB


/kaggle/working/minifasnetv2_csmr_seed100.zip